# Investment-Grade Real Estate Decision Model

This notebook is structured as an investment committee analysis: data quality, model reliability, downside risk, and capital allocation recommendation.

## 1) Data provenance and assumptions
- Macro sample: FRED-style indicators (policy rate, mortgage rate, CPI, unemployment, GDP).
- Housing sample: metro-level ZHVI/rent index style panel.
- Transaction panel is hybrid: real index anchors + structural micro assumptions.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from src.real_estate_investment_model import (
    build_hybrid_transaction_dataset, walk_forward_validation,
    default_investment_cases, monte_carlo_case, evaluate_investment_decision
)
sns.set_theme(style='whitegrid', context='talk')
Path('../results/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
df = build_hybrid_transaction_dataset(n_properties_per_metro=160, seed=42)
df[['date','metro','sale_price','annual_rent','mortgage_rate','gdp_growth']].head()

In [ ]:
df.groupby('metro')[['sale_price','annual_rent','cap_rate']].median()

## 2) Walk-forward ML validation (time-aware)
Evaluation uses expanding-window folds by year to prevent temporal leakage.

In [ ]:
model_summary, meta = walk_forward_validation(df, min_train_years=3, seed=42)
model_summary

In [ ]:
folds = meta['folds']
fig, ax = plt.subplots(figsize=(10,5))
sns.lineplot(data=folds, x='test_year', y='rmse', hue='model', marker='o', ax=ax)
ax.set_title('Walk-forward RMSE by Test Year')
fig.tight_layout(); fig.savefig('../results/figures/walk_forward_rmse.png', bbox_inches='tight')

## 3) Investment case design
We evaluate three real-world underwriting profiles: Core Stabilized, Value-Add, Opportunistic.

In [ ]:
cases = default_investment_cases()
pd.DataFrame([c.__dict__ for c in cases])[['name','purchase_price','ltv','initial_rate','annual_gross_rent','hurdle_irr']]

## 4) Correlated regime Monte Carlo
Price growth, rent shocks, and interest rates are simulated jointly with explicit correlation and regime mixture (Bull/Base/Bear).

In [ ]:
all_results = []
for c in cases:
    sim = monte_carlo_case(c, n_sims=12000, seed=41)
    decision = evaluate_investment_decision(sim, c.hurdle_irr)
    all_results.append({'case': c.name, **decision})
decision_df = pd.DataFrame(all_results)
decision_df

In [ ]:
sim_base = monte_carlo_case(cases[1], n_sims=12000, seed=99)
fig, axes = plt.subplots(1,2, figsize=(14,5))
sns.histplot(sim_base['npv'], bins=60, kde=True, ax=axes[0], color='#2a9d8f')
axes[0].axvline(sim_base['npv'].quantile(0.05), ls='--', c='black', label='VaR 5%')
axes[0].legend(); axes[0].set_title('NPV Distribution (Case B)')
sns.histplot(sim_base['irr'].dropna(), bins=60, kde=True, ax=axes[1], color='#e9c46a')
axes[1].axvline(cases[1].hurdle_irr, ls='--', c='red', label='Hurdle IRR')
axes[1].legend(); axes[1].set_title('IRR Distribution (Case B)')
fig.tight_layout(); fig.savefig('../results/figures/case_b_risk_distribution.png', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
tmp = decision_df.melt(id_vars='case', value_vars=['median_irr','prob_npv_negative','prob_irr_below_hurdle'])
sns.barplot(data=tmp, x='case', y='value', hue='variable', ax=ax)
ax.set_title('Cross-Case Risk/Return Comparison')
ax.tick_params(axis='x', rotation=20)
fig.tight_layout(); fig.savefig('../results/figures/case_comparison.png', bbox_inches='tight')

## 5) Decision framing
Translate outputs to action:
- **Invest** if median NPV > 0, downside probability controlled, and median IRR above hurdle.
- **Conditional Invest** when economics are positive but tail-risk remains elevated.
- **Do Not Invest** when downside dominates expected economics.